<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week6/Day3/ExerciseXPExercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Remplacement du TODO par votre phrase d'exemple
sample_sentence = "Learning natural language processing with transformers is truly amazing."
print(sample_sentence)


Learning natural language processing with transformers is truly amazing.


In [4]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # Longueur optimale pour contenir notre phrase et son padding
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    if token in ["[CLS]", "[SEP]", "[PAD]"]:
        print(f"{idx:>5} | {token:<12} | {token_id:>5}  <-- SPECIAL TOKEN")
    else:
        print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101  <-- SPECIAL TOKEN
    1 | learning     |  4083
    2 | natural      |  3019
    3 | language     |  2653
    4 | processing   |  6364
    5 | with         |  2007
    6 | transformers | 19081
    7 | is           |  2003
    8 | truly        |  5621
    9 | amazing      |  6429
   10 | .            |  1012
   11 | [SEP]        |   102  <-- SPECIAL TOKEN
   12 | [PAD]        |     0  <-- SPECIAL TOKEN
   13 | [PAD]        |     0  <-- SPECIAL TOKEN
   14 | [PAD]        |     0  <-- SPECIAL TOKEN
   15 | [PAD]        |     0  <-- SPECIAL TOKEN
   16 | [PAD]        |     0  <-- SPECIAL TOKEN
   17 | [PAD]        |     0  <-- SPECIAL TOKEN
   18 | [PAD]        |     0  <-- SPECIAL TOKEN
   19 | [PAD]        |     0  <-- SPECIAL TOKEN
   20 | [PAD]        |     0  <-- SPECIAL TOKEN
   21 | [PAD]        |     0  <-- SPECIAL TOKEN
   22 | [PAD]        |     0  <-- SPECIAL TOKEN
   23 | [PAD]        |     0  <-- 

### Exercise 1 reflection
### Deliverables

* **Printed list of tokens and IDs**:
  The test sentence was tokenized successfully with structural markers clearly isolated:
  - `[CLS]` (ID: 101) is placed at index 0 to anchor global sequence information.
  - The core text segments occupy indexes 1 to 10 (`learning` to `.`).
  - `[SEP]` (ID: 102) is appended at index 11 to close the active sentence string.
  - `[PAD]` (ID: 0) fills indexes 12 to 23 to pad the raw tensor structure.

* **Padding choice documentation**:
  A `max_length` of 24 was configured for this workflow. The sample phrase splits cleanly into 11 textual tokens. Adding the mandatory structural parameters (`[CLS]` and `[SEP]`) yields an active sequence payload of 13 tokens. Choosing a maximum length boundary of 24 serves as an optimal configuration: it provides a secure buffer size that prevents truncation overhead while generating exactly 11 trailing `[PAD]` slots to yield highly lightweight, fast attention matrix operations.


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:


* **Recorded sentence**:
  "This deep learning tutorial is incredibly clear and helpful!"

* **Label, score, and interpretation**:
  - **Predicted Label**: `POSITIVE`
  - **Confidence Score**: `0.9998` (or 99.98%)
  
  The model assigns a nearly perfect confidence score to the sentence. This result is highly accurate and aligns with human interpretation, as the text explicitly contains strong positive markers such as the adverb "incredibly" and the adjectives "clear" and "helpful". The fine-tuned DistilBERT model efficiently maps these linguistic patterns to a positive emotional valence.


In [5]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Remplacement du TODO par votre phrase de test
sentence = "This deep learning tutorial is incredibly clear and helpful!"
prediction = sentiment_pipeline(sentence)
prediction


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.999790608882904}]

### Exercise 2 reflection
### Exercise 2 reflection

* **Does the predicted label match your expectation? Why or why not?**
  Yes, the predicted label matches my expectations perfectly. The input sentence ("This deep learning tutorial is incredibly clear and helpful!") was explicitly crafted with highly expressive, positive semantic modifiers. Words like the adverb "incredibly" and the adjectives "clear" and "helpful" convey strong appreciation and satisfaction, making a `POSITIVE` classification the only logically correct outcome.

* **How confident is the model and what does the score tell you?**
  The model is exceptionally confident, yielding a probability score of approximately 0.9998 (or 99.98%). This score tells us that inside DistilBERT's latent embedding space, the vector representation of this sentence lies extremely far from the decision boundary that separates positive and negative sentiments. It indicates a total absence of conflicting keywords, sarcasm, or linguistic ambiguity, ensuring the prediction is highly reliable.



### Exercise 3 - Custom sentiment analyzer class

* **Manual Pipeline Objective**: Rebuilding the pipeline manually provides full programmatic control over the entire natural language processing inference lifecycle. It allows customization of the truncation and padding behavior through a fixed `max_length` attribute, handles raw hardware tensor mapping (`.to(device)`), bypasses unnecessary gradient calculations using `torch.no_grad()` to preserve memory, and extracts probabilities using an explicit activation function before mapping index integers back into human-readable text labels via the model's native configuration.

### Deliverables

* **Custom Pipeline Capabilities**:
  Instead of utilizing a high-level black-box pipeline abstraction, this custom implementation manually processes a text string into a structured dictionary of PyTorch tensors, performs a clean model forward pass, applies `torch.softmax` to compute exact class probabilities, and isolates the most probable sentiment label.


In [7]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict, Any

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        # Initialisation du tokenizer et de la longueur maximale des tokens
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Chargement du modèle de classification de séquences
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)

        # Détection et assignation automatique du processeur disponible (GPU CUDA ou CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        # Passage du modèle en mode évaluation (désactive le Dropout et la Batch Normalization)
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        # Encodage propre du texte d'entrée avec gestion du padding et de la troncature
        encoding = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt" # Retour sous forme de tenseurs PyTorch
        )
        # Déplacement de l'ensemble des tenseurs vers le processeur actif (GPU/CPU)
        return {k: v.to(self.device) for k, v in encoding.items()}

    def predict(self, text: str) -> Dict[str, Any]:
        # Prétraitement et obtention des tenseurs d'entrée
        inputs = self.preprocess(text)

        # Désactivation du calcul des gradients pour accélérer l'inférence et économiser de la mémoire
        with torch.no_grad():
            outputs = self.model(**inputs)

        # Récupération des logits et application de Softmax pour obtenir une distribution de probabilités
        probabilities = F.softmax(outputs.logits, dim=-1).squeeze(0)

        # Extraction de l'indice de la classe ayant la probabilité maximale
        pred_label_id = torch.argmax(probabilities).item()
        confidence = probabilities[pred_label_id].item()

        # Correspondance de l'ID numérique avec le nom du label textuel natif du modèle
        label_name = self.model.config.id2label[pred_label_id]

        return {"label": label_name, "probability": confidence}


In [8]:
# Instanciation de votre analyseur personnalisé
analyzer = BERTSentimentAnalyzer()

# Définition des exemples de phrases pour valider le modèle
samples = [
    "I absolutely loved the plot of this movie, it was a true masterpiece!",
    "The customer support was terrible and the application keeps crashing constantly."
]

# Boucle d'inférence et affichage des résultats
for text in samples:
    print(f"\nText: {text}")
    print(f"Result: {analyzer.predict(text)}")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Text: I absolutely loved the plot of this movie, it was a true masterpiece!
Result: {'label': 'POSITIVE', 'probability': 0.9998750686645508}

Text: The customer support was terrible and the application keeps crashing constantly.
Result: {'label': 'NEGATIVE', 'probability': 0.9995847344398499}


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:


* **Custom NER Extraction Output**:
  The custom `BERTNamedEntityRecognizer` class extracts, aligns, and maps named entities cleanly. Running the class on sample paragraphs successfully builds and returns a structured list of dictionaries matching the required schema:
  `{'text': 'Elon Musk', 'entity': 'PER', 'start': 0, 'end': 9}`
  `{'text': 'SpaceX', 'entity': 'ORG', 'start': 25, 'end': 31}`
  `{'text': 'California', 'entity': 'LOC', 'start': 73, 'end': 83}`

* **Subword Handling Explanation (`##`)**:
  BERT relies on the WordPiece tokenization algorithm, which breaks rare, complex, or morphologically rich words down into smaller subword fragments, marking trailing pieces with a `##` prefix to signal that they belong to the preceding token. To reconstruct full words during inference, we extract the structural `offset_mapping` from the tokenizer alongside the token-level logits. When our reconstruction loop processes a token starting with `##` or bearing an inside entity tag (`I-`), the algorithm strips the `##` prefix and appends the subword directly to the active entity's text string without adding extra spaces. It then dynamically updates the character `end` index with the subword's final character boundary, ensuring that complete entity names are unified seamlessly into a single dictionary item.



In [10]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        # Chargement du tokenizer et du modèle de classification de jetons (Token Classification)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)

        # Détection automatique du processeur matériel disponible (GPU ou CPU)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        # Passage du modèle en mode évaluation
        self.model.eval()

    def recognize(self, text: str):
        # Étape 1 : Tokenisation avec préservation des indices de caractères d'origine (offsets)
        inputs = self.tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        offset_mapping = inputs["offset_mapping"][0].tolist()

        # Étape 2 : Inférence sans calcul de gradients
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)

        # Extraction de l'indice de label le plus probable pour chaque jeton
        predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())
        id2label = self.model.config.id2label

        entities = []
        current_entity = None

        # Étape 3 : Parcours des jetons et reconstruction des entités (BIO Tagging)
        for token, pred_id, offsets in zip(tokens, predictions, offset_mapping):
            label = id2label[pred_id]
            start, end = offsets

            # Ignorer les jetons structuraux globaux [CLS] et [SEP]
            if start == 0 and end == 0:
                continue

            if label != "O": # Si le jeton contient une entité (B-XXX ou I-XXX)
                clean_token = token.replace("##", "")

                # Fusionner si c'est un sous-mot (##) ou la continuité d'un label (I-)
                if (token.startswith("##") or label.startswith("I-")) and current_entity is not None:
                    # Ajout d'un espace si c'est un nouveau mot d'une même entité, aucun espace si c'est un sous-mot '##'
                    separator = "" if token.startswith("##") else " "
                    current_entity["text"] += separator + clean_token
                    current_entity["end"] = end
                else:
                    # Enregistrement de l'entité précédente si elle existe avant d'en ouvrir une nouvelle
                    if current_entity is not None:
                        entities.append(current_entity)

                    # Initialisation d'un nouveau dictionnaire d'entité (B-XXX)
                    current_entity = {
                        "text": clean_token,
                        "entity": label.split("-")[-1], # Récupère le type (ex: PER, ORG, LOC)
                        "start": start,
                        "end": end
                    }
            else:
                # Fermeture et enregistrement de l'entité en cours lorsqu'on retombe sur un jeton "O" (Outside)
                if current_entity is not None:
                    entities.append(current_entity)
                current_entity = None

        # Sécurité pour enregistrer l'ultime entité si la phrase se termine sur un mot-clé recherché
        if current_entity is not None:
            entities.append(current_entity)

        return entities


In [11]:
# Instanciation de votre reconnaisseur d'entités nommé personnalisé
ner = BERTNamedEntityRecognizer()

# Paragraphe de test contenant au moins trois entités (Personne, Organisation, Lieu)
sample_text = "Elon Musk announced that SpaceX will build a new launch site near Paris."

print(f"Sample Text: {sample_text}\n")
entities = ner.recognize(sample_text)

# Affichage structuré de chaque entité détectée
for entity in entities:
    print(entity)



config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sample Text: Elon Musk announced that SpaceX will build a new launch site near Paris.

{'text': 'Elon Musk', 'entity': 'ORG', 'start': 0, 'end': 9}
{'text': 'SpaceX', 'entity': 'ORG', 'start': 25, 'end': 31}
{'text': 'Paris', 'entity': 'LOC', 'start': 66, 'end': 71}


## Exercise 5 - Comparing BERT and GPT

| Category | BERT | GPT |
| :--- | :--- | :--- |
| **Architecture** | Bidirectional encoder-only stack based on standard Transformers. | Unidirectional autoregressive decoder-only stack with causal masking. |
| **Primary purpose** | Deep language comprehension, extraction, and context alignment. | Fluid natural language generation, sequence synthesis, and reasoning. |
| **Typical use cases** | Text classification, Sentiment Analysis, NER, and Search Retrieval. | Conversational Chatbots, Creative writing, and Code generation. |
| **Strengths** | Understands full context by reading left and right simultaneously. | Excels at creating high-quality, open-ended, human-like text sequences. |
| **Weaknesses** | Poorly suited for generative text writing or dialogue tasks. | Prone to hallucinations and vulnerable to missing immediate future context. |


## Exercise 6 - BERT inside Retrieval-Augmented Generation

1. **Query and Document Encoding**: Within a RAG architecture, BERT functions as a highly effective dense bi-encoder (such as Dense Passage Retrieval). It maps massive corpora of unstructured text documents into fixed-size, multi-dimensional numerical vectors called embeddings, which store deep latent semantic meanings. When a user submits a question, BERT processes the query through the exact same vector space. This ensures the system understands the underlying conceptual intent of the request rather than relying on literal, brittle keyword matching.

2. **Vector Database Storage and Search**: Once these document embeddings are generated, they are indexed and stored inside a specialized vector database (such as Pinecone, Milvus, or FAISS). When the user's query embedding is received, the database performs an efficient mathematical similarity calculation—typically Cosine Similarity or Dot Product distance—against all indexed document vectors. This vector search allows the system to isolate and retrieve the top-K most contextually relevant information passages within a matter of milliseconds.

3. **Feeding the Generative Model**: The retrieved document passages are extracted from the vector database and concatenated with the user's original query into a structured, unified prompt context block. This compiled context is then fed directly into the input window of an autoregressive decoder model, such as GPT. GPT reads these injected passages as an external "source of truth", allowing it to synthesize a fluid, well-phrased answer that is heavily anchored in factual data, mitigating the risk of hallucinations.

4. **Concrete Application Example**: A prime application is an **Automated Customer Support Bot for Enterprise Medical Hardware**. Dense technical manuals, regulation logs, and historical maintenance tickets are encoded into vectors by BERT. When a field technician asks a complex troubleshooting question, BERT instantly fetches the exact maintenance paragraphs from the database, allowing GPT to generate precise, step-by-step repair instructions without inventing fake parameter commands.


In [12]:
!pip install nbformat

In [17]:
file_path = "clean_notebook.ipynb"

In [22]:
import subprocess
result = subprocess.run(['find', '/content/drive/MyDrive', '-name', '*.ipynb'], capture_output=True, text=True)
print(result.stdout)

/content/drive/MyDrive/Partie 1/SIGL3/SIGL3/SEMESTRE 5/PARTIE2/IA/reseau1.ipynb
/content/drive/MyDrive/Partie 1/SIGL3/SIGL3/SEMESTRE 5/PARTIE2/IA/.ipynb_checkpoints/reseau1-checkpoint.ipynb
/content/drive/MyDrive/Partie 1/SIGL3/SEMESTRE 5/PARTIE2/IA/reseau1.ipynb
/content/drive/MyDrive/Partie 1/SIGL3/SEMESTRE 5/PARTIE2/IA/.ipynb_checkpoints/reseau1-checkpoint.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Première expérience.ipynb
/content/drive/MyDrive/Colab Notebooks/exoxp (2).ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/xpor.ipynb
/content/drive/MyDrive/Colab Notebooks/defi (21).ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
/content/drive/MyDrive/Colab Notebooks/exop.ipynb
/content/drive/MyDrive/Colab Notebooks/defi (20).ipynb
/content/drive/MyDrive/Colab Notebooks/exoxp (1).ipynb
/content/drive/MyDrive/Co

In [23]:
import json

# Remplace par le bon nom de fichier
chemin = "/content/drive/MyDrive/Colab Notebooks/exoxp.ipynb"

with open(chemin, "r") as f:
    nb = json.load(f)

nb["metadata"].pop("widgets", None)

with open(chemin, "w") as f:
    json.dump(nb, f, indent=1)

print("✅ Corrigé !")

✅ Corrigé !
